# Construcao do Dataset — LRG

Pipeline completo para construir o HDF5 com espectros de **Luminous Red
Galaxies (LRGs)** do eBOSS/SDSS DR17.

**Etapas:**
1. Combinar catalogos NGC + SGC (clustering)
2. Cross-match com o catalogo full
3. Ler espectros do SDSS (com cache local) e calcular S/N
4. Salvar HDF5 estruturado

**Estrutura do HDF5 gerado (`data/processed/LRG/LRGspectra_raw.h5`):**

```
catalog/                          <- arrays flat (N,)
  ├── spec_id           (str)     "LRG_000042"
  │
  ├── plate, mjd, fiberid (int)
  ├── ra, dec           (float)
  ├── spectype, subtype (str)
  │
  ├── redshift          (float)   target
  ├── zerr, chi2, deltachi2 (float)
  ├── zwarning          (int)
  │
  ├── npixels           (int)
  ├── sn_median         (float)   computado por nos
  ├── sn_median_all     (float)   do pipeline
  │
  # Fotometria optica (MODELMAG vetor [u,g,r,i,z])
  ├── g_mag, r_mag      (float)
  ├── i_mag, z_mag      (float)
  ├── gr_color, ri_color (float)
  │
  # Fotometria infravermelha (WISE)
  ├── w1_mag            (float)
  ├── w1_w2_color       (float)
  │
  └── line_flux         (float)   NaN para LRG (sem emissao forte)
```

> **Nota:** o catalogo eBOSS LRG DR16 ja usa redrock como pipeline padrao —
> `Z` e `Z_REDROCK` sao **identicos**. Por isso nao salvamos os campos
> `redshift_redrock`, `zwarning_redrock`, `spectype_redrock` (seriam redundantes).


## 1. Imports e Configuracao

In [ ]:
import os
import sys
import time
import hashlib
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import h5py
from tqdm import tqdm
from astropy.io import fits
from astropy.table import Table
from astropy import units as u
from astropy.coordinates import SkyCoord

# Adiciona raiz do projeto ao sys.path para importar config
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR
while not (PROJECT_ROOT / "config.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import paths_for, print_info

OBJECT_TYPE = "LRG"
print_info()

paths = paths_for(OBJECT_TYPE)
print(f"\nObjeto: {OBJECT_TYPE}")
print(f"NGC FITS:      {paths['ngc_fits']}")
print(f"SGC FITS:      {paths['sgc_fits']}")
print(f"FULL FITS:     {paths['full_fits']}")
print(f"Combined out:  {paths['combined_fits']}")
print(f"Filtered out:  {paths['filtered_fits']}")
print(f"HDF5 out:      {paths['spectra_h5']}")
print(f"Cache:         {paths['cache_dir']}")

# Cria diretorios necessarios
paths["cache_dir"].mkdir(parents=True, exist_ok=True)
paths["spectra_h5"].parent.mkdir(parents=True, exist_ok=True)
paths["filtered_fits"].parent.mkdir(parents=True, exist_ok=True)


## 2. Inspecionar colunas dos FITS

Antes de processar, dar uma olhada nas colunas disponiveis em cada FITS.
Util pra saber o que vai pra `catalog/` no HDF5.


In [ ]:
def inspect_fits(path, max_cols=20):
    print(f"\n=== {path.name} ===")
    with fits.open(path) as h:
        d = h[1].data
        print(f"  N rows: {len(d):,}")
        names = d.dtype.names
        print(f"  N cols: {len(names)}")
        print(f"  Cols (primeiras {max_cols}): {names[:max_cols]}")

inspect_fits(paths['ngc_fits'])
inspect_fits(paths['sgc_fits'])
inspect_fits(paths['full_fits'], max_cols=30)


## 3. Combinar NGC + SGC

Os catalogos clustering vem separados por hemisferio (NGC/SGC). Concatena
ambos em um unico FITS combinado.


In [ ]:
def combine_ngc_sgc(ngc_path, sgc_path, out_path):
    if out_path.exists():
        print(f"Combinado ja existe: {out_path}")
        with fits.open(out_path) as h:
            return Table(h[1].data)

    with fits.open(ngc_path) as h:
        ngc = Table(h[1].data)
    with fits.open(sgc_path) as h:
        sgc = Table(h[1].data)

    print(f"NGC: {len(ngc):,} | SGC: {len(sgc):,}")
    combined = Table(np.hstack([ngc, sgc]))
    combined.write(out_path, format="fits", overwrite=True)
    print(f"Combinado: {len(combined):,} -> {out_path}")
    return combined


clustering = combine_ngc_sgc(paths['ngc_fits'], paths['sgc_fits'], paths['combined_fits'])
print(f"\nColunas: {clustering.colnames[:15]}...")


## 4. Cross-match com catalogo full

O catalogo `clustering` tem RA/DEC mas faltam alguns metadados (magnitudes,
etc.) que estao no catalogo `full`. Fazemos um match angular de 1 arcsec
para anexar essas colunas.


In [ ]:
def cross_match(clustering, full_path, out_path, max_sep_arcsec=1.0):
    if out_path.exists():
        print(f"Filtrado ja existe: {out_path}")
        with fits.open(out_path) as h:
            return Table(h[1].data)

    with fits.open(full_path) as h:
        full = Table(h[1].data)
    print(f"Full catalog: {len(full):,} objetos")

    cl_coords   = SkyCoord(ra=clustering['RA'] * u.deg, dec=clustering['DEC'] * u.deg)
    full_coords = SkyCoord(ra=full['RA'] * u.deg,        dec=full['DEC'] * u.deg)

    idx, d2d, _ = cl_coords.match_to_catalog_sky(full_coords)
    mask = d2d < max_sep_arcsec * u.arcsec
    filtered = full[idx[mask]]

    filtered.write(out_path, format="fits", overwrite=True)
    print(f"Match (<{max_sep_arcsec}\"): {len(filtered):,}/{len(clustering):,} -> {out_path}")
    return filtered


filtered = cross_match(clustering, paths['full_fits'], paths['filtered_fits'])

import pandas as pd
pd.set_option("display.max_columns", None)
scalar_cols = [c for c in filtered.colnames if len(filtered[c].shape) <= 1]
df_filtered = filtered[scalar_cols].to_pandas()
df_filtered.head()


## 5. Funcao para ler espectros (com cache)

Le um espectro do disco se ja foi baixado, senao baixa do SDSS DR17.
Cache local evita re-download em runs futuros.


In [ ]:
def get_cache_path(cache_dir, plate, mjd, fiber):
    """Caminho deterministico do cache (hash MD5)."""
    key = f"{plate}-{mjd}-{fiber}"
    h = hashlib.md5(key.encode()).hexdigest()
    return cache_dir / f"{h}.fits"


def candidate_urls(plate, mjd, fiber):
    """URLs do SDSS DR17 a tentar em ordem.

    QSOs do DR16Q incluem espectros legados (SDSS-I/II, plate < 3500),
    cuja estrutura no SAS e diferente do eBOSS.
    """
    urls = []
    # eBOSS / BOSS reductions (DR17)
    for v in ["v5_13_2", "v5_13_0", "v5_10_10", "v5_7_0"]:
        urls.append(
            f"https://data.sdss.org/sas/dr17/eboss/spectro/redux/{v}/"
            f"spectra/lite/{plate}/spec-{plate}-{mjd:05d}-{fiber:04d}.fits"
        )
    # SDSS-I/II legacy (run2d=26, plate em 4-digit zero-padded)
    urls.append(
        f"https://data.sdss.org/sas/dr17/sdss/spectro/redux/26/"
        f"spectra/lite/{plate:04d}/spec-{plate:04d}-{mjd:05d}-{fiber:04d}.fits"
    )
    for run2d in ["103", "104"]:
        urls.append(
            f"https://data.sdss.org/sas/dr17/sdss/spectro/redux/{run2d}/"
            f"spectra/lite/{plate:04d}/spec-{plate:04d}-{mjd:05d}-{fiber:04d}.fits"
        )
    return urls


def read_spec(plate, mjd, fiber, cache_dir, max_attempts=2, delay=1):
    """Le um espectro (cache > download). Retorna (wave, flux, sigma) ou Nones se falhar."""
    cache_path = get_cache_path(cache_dir, plate, mjd, fiber)

    # Tenta cache
    if cache_path.exists():
        try:
            with fits.open(cache_path) as f:
                d = f[1].data
                W = 10 ** d['loglam']
                F = d['flux'] * 1e-17
                iVar = d['ivar']
                ok = np.argwhere(iVar > 0).flatten()
                if len(ok) == 0:
                    return None, None, None
                return W[ok], F[ok], 1e-17 / np.sqrt(iVar[ok])
        except Exception:
            cache_path.unlink(missing_ok=True)

    # Tenta cada URL candidata
    for url in candidate_urls(plate, mjd, fiber):
        for attempt in range(max_attempts):
            try:
                with fits.open(url) as f:
                    d = f[1].data
                    f.writeto(cache_path, overwrite=True)
                W = 10 ** d['loglam']
                F = d['flux'] * 1e-17
                iVar = d['ivar']
                ok = np.argwhere(iVar > 0).flatten()
                if len(ok) == 0:
                    return None, None, None
                return W[ok], F[ok], 1e-17 / np.sqrt(iVar[ok])
            except Exception as e:
                if "404" in str(e):
                    break  # passa pra proxima URL
                time.sleep(delay)

    return None, None, None


## 6. Detectar nomes de colunas

Os FITS do SDSS as vezes usam variantes de nome (`PLATE` vs `plate`,
`Z` vs `z_pipeline`). Detectamos automaticamente.


In [ ]:
# Nomes exatos das colunas no FITS full do eBOSS LRG DR16
COL_PLATE             = "PLATE"           # uppercase em LRG (em ELG e lowercase!)
COL_MJD               = "MJD"
COL_FIBERID           = "FIBERID"
COL_RA                = "RA"
COL_DEC               = "DEC"
COL_SPECTYPE          = "CLASS"           # CLASS em LRG (SPECTYPE em ELG)
COL_SUBTYPE           = "SUBTYPE"
COL_REDSHIFT          = "Z"
COL_ZERR              = "ZERR"
COL_CHI2              = "CHI2"
COL_DELTACHI2         = "DELTACHI2"
COL_ZWARNING          = "ZWARNING"        # ZWARNING em LRG (ZWARN em ELG)
COL_NPIXELS           = "NPIXELS"
COL_SN_MEDIAN_ALL     = "SN_MEDIAN_ALL"

# Fotometria optica (MODELMAG e um vetor 5D: u, g, r, i, z)
COL_MODELMAG          = "MODELMAG"

# Fotometria infravermelha (WISE)
COL_W1_MAG            = "W1_MAG"
COL_W1_NMAG           = "W1_NANOMAGGIES"
COL_W2_NMAG           = "W2_NANOMAGGIES"

# LRG nao tem linha de emissao forte (sera NaN)
COL_LINE_FLUX         = ""

# Sanity check
required = [COL_PLATE, COL_MJD, COL_FIBERID, COL_RA, COL_DEC, COL_REDSHIFT]
missing = [c for c in required if c not in filtered.colnames]
if missing:
    raise ValueError(f"Colunas obrigatorias nao encontradas: {missing}")

optional = [COL_SPECTYPE, COL_SUBTYPE, COL_ZERR, COL_CHI2, COL_DELTACHI2,
            COL_ZWARNING, COL_NPIXELS, COL_SN_MEDIAN_ALL,
            COL_MODELMAG, COL_W1_MAG, COL_W1_NMAG, COL_W2_NMAG]
missing_opt = [c for c in optional if c and c not in filtered.colnames]
if missing_opt:
    print(f"[aviso] Colunas opcionais ausentes (serao NaN): {missing_opt}")
else:
    print("Todas as colunas encontradas.")

if COL_MODELMAG in filtered.colnames:
    print(f"MODELMAG shape: {filtered[COL_MODELMAG].shape}  (esperado: (N, 5))")


## 7. Filtrar entradas validas

Aplica cortes de qualidade:
- `Z >= 0` (redshift fisico)
- `ZWARNING == 0` se quiser apenas confiaveis (controlado por flag)


In [ ]:
USE_SPECTYPE_CUT  = True               # filtra so o tipo esperado
USE_ZWARNING_CUT  = False              # True = mantem so zwarning == 0
EXPECTED_SPECTYPE = "GALAXY"             # tipo esperado para esta classe
N_LIMIT           = None               # None = todos. Para teste rapido use 1000.

mask = filtered[COL_REDSHIFT] >= 0
print(f"Apos z >= 0:                {mask.sum():,}/{len(filtered):,}")

if USE_SPECTYPE_CUT:
    # SPECTYPE pode vir como bytes ou str — normaliza para comparar
    spectypes = np.array([str(s).strip() for s in filtered[COL_SPECTYPE]])
    n_before = mask.sum()
    mask &= (spectypes == EXPECTED_SPECTYPE)
    n_removed = n_before - mask.sum()
    print(f"Apos SPECTYPE == '{EXPECTED_SPECTYPE}': {mask.sum():,} "
          f"(removidos {n_removed} de outros tipos)")

if USE_ZWARNING_CUT:
    mask &= filtered[COL_ZWARNING] == 0
    print(f"Apos ZWARNING == 0:         {mask.sum():,}")

cat = filtered[mask]
if N_LIMIT is not None:
    cat = cat[:N_LIMIT]
    print(f"Limitado para {N_LIMIT:,} (TESTE)")

print(f"\nFinal: {len(cat):,} objetos a processar")


## 8. Baixar/ler espectros em paralelo

Usa `ProcessPoolExecutor` para paralelizar. Ajuste `MAX_WORKERS` conforme
a maquina (default = numero de CPUs).


In [ ]:
MAX_WORKERS = max(4, os.cpu_count() - 2)
print(f"Workers: {MAX_WORKERS}")

# Prepara lista de tarefas
tasks = [(int(cat[COL_PLATE][i]),
          int(cat[COL_MJD][i]),
          int(cat[COL_FIBERID][i]),
          paths["cache_dir"]) for i in range(len(cat))]


def worker(args):
    """Le espectro e computa S/N mediano."""
    plate, mjd, fiber, cache_dir = args
    W, F, S = read_spec(plate, mjd, fiber, cache_dir)
    if W is None:
        return (plate, mjd, fiber, None, None, None, None)
    # S/N mediano: F / sigma = flux * sqrt(ivar)
    sn = float(np.median(F / S)) if len(F) else float("nan")
    return (plate, mjd, fiber, W, F, S, sn)


# Testa com 1 espectro primeiro
print("\nTeste com 1 espectro:")
test_plate, test_mjd, test_fiber, _ = tasks[0]
W, F, S = read_spec(test_plate, test_mjd, test_fiber, paths["cache_dir"])
if W is None:
    print(f"  [FALHA] {test_plate}-{test_mjd}-{test_fiber}")
    raise RuntimeError("Teste falhou — verifique conectividade ou primeiro espectro")
print(f"  [OK] {test_plate}-{test_mjd}-{test_fiber}: {len(W)} pts, "
      f"{W[0]:.0f}-{W[-1]:.0f} A, S/N med = {np.median(F / S):.2f}")


In [ ]:
# Loop principal — processa todos os espectros
results = {}  # spec_idx -> (wave, flux, sigma, sn_median)
fails = 0

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(worker, tasks[i]): i for i in range(len(tasks))}
    for fut in tqdm(as_completed(futures), total=len(tasks), desc="Espectros"):
        i = futures[fut]
        try:
            plate, mjd, fiber, W, F, S, sn = fut.result()
            if W is None:
                fails += 1
                continue
            results[i] = (W.astype(np.float32),
                          F.astype(np.float32),
                          S.astype(np.float32),
                          sn)
        except Exception as e:
            fails += 1
            print(f"[erro] indice {i}: {e}")

print(f"\nSucesso: {len(results):,}/{len(tasks):,}")
print(f"Falhas:  {fails:,}")


## 9. Salvar HDF5 estruturado

In [ ]:
with h5py.File(paths["spectra_h5"], "w") as f:
    valid_idx = sorted(results.keys())
    N = len(valid_idx)
    print(f"Salvando {N:,} espectros em {paths['spectra_h5']}")

    spec_ids = [f"{OBJECT_TYPE}_{i:06d}" for i in range(N)]

    def col_or_nan(col_name, dtype=np.float32):
        if not col_name or col_name not in filtered.colnames:
            return np.full(N, np.nan, dtype=dtype)
        return np.array([cat[col_name][i] for i in valid_idx], dtype=dtype)

    def col_or_empty_str(col_name):
        if not col_name or col_name not in filtered.colnames:
            return np.array([""] * N, dtype="S20")
        return np.array([str(cat[col_name][i]).strip() for i in valid_idx], dtype="S20")

    def modelmag_band(band_idx, dtype=np.float32):
        """Extrai uma banda de MODELMAG (vetor [u, g, r, i, z]: 0=u, 1=g, 2=r, 3=i, 4=z)."""
        if COL_MODELMAG not in filtered.colnames:
            return np.full(N, np.nan, dtype=dtype)
        return np.array([cat[COL_MODELMAG][i, band_idx] for i in valid_idx], dtype=dtype)

    cat_grp = f.create_group("catalog")

    cat_grp.create_dataset("spec_id",          data=np.array(spec_ids, dtype="S20"))

    # Identificacao
    cat_grp.create_dataset("plate",            data=col_or_nan(COL_PLATE, np.int32))
    cat_grp.create_dataset("mjd",              data=col_or_nan(COL_MJD, np.int32))
    cat_grp.create_dataset("fiberid",          data=col_or_nan(COL_FIBERID, np.int32))

    # Posicao
    cat_grp.create_dataset("ra",               data=col_or_nan(COL_RA, np.float64))
    cat_grp.create_dataset("dec",              data=col_or_nan(COL_DEC, np.float64))

    # Classificacao
    cat_grp.create_dataset("spectype",         data=col_or_empty_str(COL_SPECTYPE))
    cat_grp.create_dataset("subtype",          data=col_or_empty_str(COL_SUBTYPE))

    # Redshift e qualidade
    cat_grp.create_dataset("redshift",         data=col_or_nan(COL_REDSHIFT, np.float32))
    cat_grp.create_dataset("zerr",             data=col_or_nan(COL_ZERR, np.float32))
    cat_grp.create_dataset("chi2",             data=col_or_nan(COL_CHI2, np.float32))
    cat_grp.create_dataset("deltachi2",        data=col_or_nan(COL_DELTACHI2, np.float32))
    cat_grp.create_dataset("zwarning",         data=col_or_nan(COL_ZWARNING, np.int32))

    # Qualidade do espectro
    cat_grp.create_dataset("npixels",          data=col_or_nan(COL_NPIXELS, np.int32))
    cat_grp.create_dataset("sn_median",        data=np.array([results[i][3] for i in valid_idx],
                                                              dtype=np.float32))
    cat_grp.create_dataset("sn_median_all",    data=col_or_nan(COL_SN_MEDIAN_ALL, np.float32))

    # Fotometria optica (de MODELMAG)
    g_arr = modelmag_band(1)
    r_arr = modelmag_band(2)
    i_arr = modelmag_band(3)
    z_arr = modelmag_band(4)
    cat_grp.create_dataset("g_mag",            data=g_arr)
    cat_grp.create_dataset("r_mag",            data=r_arr)
    cat_grp.create_dataset("i_mag",            data=i_arr)
    cat_grp.create_dataset("z_mag",            data=z_arr)
    cat_grp.create_dataset("gr_color",         data=(g_arr - r_arr).astype(np.float32))
    cat_grp.create_dataset("ri_color",         data=(r_arr - i_arr).astype(np.float32))

    # Fotometria IR (WISE)
    w1_arr = col_or_nan(COL_W1_MAG, np.float32)
    cat_grp.create_dataset("w1_mag",           data=w1_arr)
    w1n = col_or_nan(COL_W1_NMAG, np.float64)
    w2n = col_or_nan(COL_W2_NMAG, np.float64)
    with np.errstate(invalid="ignore", divide="ignore"):
        w1w2 = (-2.5 * np.log10(w1n / w2n)).astype(np.float32)
    cat_grp.create_dataset("w1_w2_color",      data=w1w2)

    # Linha de emissao (LRG: NaN)
    line_ds = cat_grp.create_dataset("line_flux", data=col_or_nan(COL_LINE_FLUX, np.float32))
    line_ds.attrs["line_name"] = "(none)"

    # ---- spectra/ : 1 grupo por espectro com wave, flux, ivar ----
    spec_grp = f.create_group("spectra")
    for new_i, orig_i in enumerate(tqdm(valid_idx, desc="Escrevendo HDF5")):
        sid = spec_ids[new_i]
        W, F, S, _ = results[orig_i]
        g = spec_grp.create_group(sid)
        g.create_dataset("wave", data=W, compression="gzip", compression_opts=4)
        g.create_dataset("flux", data=F, compression="gzip", compression_opts=4)
        g.create_dataset("ivar", data=(1.0 / (S ** 2)).astype(np.float32),
                         compression="gzip", compression_opts=4)

print(f"\nHDF5 escrito: {paths['spectra_h5']} "
      f"({paths['spectra_h5'].stat().st_size / 1e9:.2f} GB)")


## 10. Verificacao

Reabre o HDF5 e checa estrutura.


In [13]:
with h5py.File(paths["spectra_h5"], "r") as f:
    cat = f["catalog"]
    print(f"Grupos: {list(f.keys())}")
    print(f"\ncatalog/ datasets ({len(cat)}): {list(cat.keys())}")
    print(f"  N: {len(cat['spec_id']):,}")
    print(f"  z range:        {cat['redshift'][:].min():.3f} - {cat['redshift'][:].max():.3f}")
    print(f"  S/N mediano:    {np.nanmedian(cat['sn_median'][:]):.2f} (computado)")
    print(f"  S/N pipeline:   {np.nanmedian(cat['sn_median_all'][:]):.2f}")

    # Optica
    print(f"\n  Fotometria optica (MODELMAG):")
    print(f"    g_mag:        {np.nanmin(cat['g_mag'][:]):.2f} - {np.nanmax(cat['g_mag'][:]):.2f}")
    print(f"    r_mag:        {np.nanmin(cat['r_mag'][:]):.2f} - {np.nanmax(cat['r_mag'][:]):.2f}")
    print(f"    gr_color:     {np.nanmin(cat['gr_color'][:]):.2f} - {np.nanmax(cat['gr_color'][:]):.2f}")
    print(f"    ri_color:     {np.nanmin(cat['ri_color'][:]):.2f} - {np.nanmax(cat['ri_color'][:]):.2f}")

    # IR
    print(f"\n  Fotometria IR (WISE):")
    print(f"    w1_mag:       {np.nanmin(cat['w1_mag'][:]):.2f} - {np.nanmax(cat['w1_mag'][:]):.2f}")
    print(f"    w1_w2_color:  {np.nanmin(cat['w1_w2_color'][:]):.2f} - {np.nanmax(cat['w1_w2_color'][:]):.2f}")

    n_zwarn = int((cat['zwarning'][:] != 0).sum())
    print(f"\n  ZWARNING != 0:  {n_zwarn} ({100*n_zwarn/len(cat['zwarning']):.1f}%)")

    types, counts = np.unique(cat['spectype'][:], return_counts=True)
    print(f"\n  spectype: {dict(zip([t.decode() if isinstance(t, bytes) else t for t in types], counts))}")

    print(f"\nspectra/: {len(f['spectra']):,} grupos")
    sample_id = list(f["spectra"].keys())[0]
    g = f[f"spectra/{sample_id}"]
    print(f"  Exemplo {sample_id}: wave {g['wave'].shape}  "
          f"({g['wave'][0]:.0f}-{g['wave'][-1]:.0f} A)")


Grupos: ['catalog', 'spectra']

catalog/ datasets (28): ['chi2', 'dec', 'deltachi2', 'fiberid', 'g_mag', 'gr_color', 'i_mag', 'line_flux', 'mjd', 'npixels', 'plate', 'r_mag', 'ra', 'redshift', 'redshift_redrock', 'ri_color', 'sn_median', 'sn_median_all', 'spec_id', 'spectype', 'spectype_redrock', 'subtype', 'w1_mag', 'w1_w2_color', 'z_mag', 'zerr', 'zwarning', 'zwarning_redrock']
  N: 100
  z range:        0.604 - 0.993
  S/N mediano:    1.15 (computado)
  S/N pipeline:   1.17

  Fotometria optica (MODELMAG):
    g_mag:        21.22 - 26.38
    r_mag:        21.09 - 23.30
    gr_color:     -0.18 - 4.72
    ri_color:     0.99 - 2.04

  Fotometria IR (WISE):
    w1_mag:       14.99 - 16.71
    w1_w2_color:  -2.31 - 0.67

  ZWARNING != 0:  19 (19.0%)

  spectype: {'GALAXY': 100}

spectra/: 100 grupos
  Exemplo LRG_000000: wave (4568,)  (3595-10332 A)
